# Week 2: ML Task Framing (FlyRank AI)

**Lane:** Classification  
**Deliverable:** `work/notebooks/w02_ml_task_framing.ipynb`

This notebook frames my lane as a well-defined ML problem: the task type, the target, the success metric, the unit of analysis, and the case for ML over a fixed rule.

## 1. My lane as an ML task (type)

**Task type: supervised binary classification.**

For each content item (for example, an article or page) we predict whether it will be a **positive outcome** (a high-performing item) or not, using only features known at or before publication, such as length, structure, and link counts.

**Why classification?** The decision we want to support is a yes/no triage call: *prioritize this item or not*. The label is discrete (`0` or `1`), historical examples with known outcomes exist, and the model should output a probability so the team can rank items and choose a threshold that fits the available capacity.

## 2. Target or proxy

- **Target column:** `is_target_positive`
- **`1`** = the content item reached the performance goal within 90 days of publication (here: ranked in the Google top 10 for its primary keyword).
- **`0`** = it did not.

**Why this is a proxy.** The ultimate business outcome is value: traffic, leads, and revenue. Those are noisy, delayed, and hard to attribute to a single item. A top-10 ranking is observable, timely, and strongly correlated with traffic. It is therefore a practical stand-in for "this content worked".

**Caveats.** The proxy can diverge from real value (a page can rank well without converting), and the 90-day threshold is a design choice. Both should be revisited as we learn more. Outcome columns are kept **out of the feature set** to avoid target leakage.

## 3. Success metric

**Primary metrics: ROC-AUC and Precision-Recall AUC (PR-AUC).**

- **ROC-AUC** measures how well the model ranks positives above negatives across all thresholds (0.5 = random, 1.0 = perfect). It is threshold-independent and insensitive to the chosen cutoff.
- **PR-AUC** (average precision) focuses on the positive class: how many flagged items are truly positive (precision) versus how many positives we find (recall). It is more informative than ROC-AUC when positives are rare.

**Why not accuracy?** If only ~20% of items are positive, a model that predicts `0` for everything scores ~80% accuracy while being useless. Accuracy also hides the precision/recall trade-off and depends on a fixed 0.5 threshold. The cell below demonstrates this.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, roc_auc_score, average_precision_score

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)

RNG_SEED = 42
rng = np.random.default_rng(RNG_SEED)

## 4. The unit of analysis, as a real dataframe

**One row = one unique content item.** Each row has a unique `content_id`, pre-publication features, and the outcome used to build the target. The data below is a **mock slice** standing in for our lane's starter data; to use the real file, swap the generator for `pd.read_csv(...)`.

In [ ]:
n = 1000

df = pd.DataFrame({
    "content_id": [f"C{i:04d}" for i in range(1, n + 1)],
    "content_type": rng.choice(["blog", "guide", "landing_page", "news"], size=n, p=[0.45, 0.2, 0.2, 0.15]),
    "word_count": np.clip(rng.lognormal(mean=7.0, sigma=0.45, size=n), 200, 6000).round().astype(int),
    "title_length": rng.integers(20, 90, size=n),
    "num_headings": rng.integers(0, 16, size=n),
    "num_images": rng.integers(0, 12, size=n),
    "internal_links": rng.poisson(5, size=n),
    "backlinks": rng.poisson(3, size=n),
    "readability_score": np.clip(rng.normal(60, 12, size=n), 15, 100).round(1),
})

# Mock outcome: ranked in the top 10 within 90 days (a noisy function of the features).
# In real data this column is observed, not simulated.
logit = (
    -2.6
    + 0.0007 * (df["word_count"] - 1200)
    + 0.10 * df["num_headings"]
    + 0.15 * df["backlinks"]
    + 0.04 * df["internal_links"]
    + 0.02 * (df["readability_score"] - 60)
    + df["content_type"].map({"guide": 0.5, "blog": 0.1, "landing_page": -0.2, "news": -0.4})
)
prob = 1 / (1 + np.exp(-logit))
df["top10_within_90d"] = rng.binomial(1, prob)

# Target column: binary proxy label
df["is_target_positive"] = df["top10_within_90d"].astype(int)

OUTCOME_COLS = ["top10_within_90d"]
ID_COL, TARGET = "content_id", "is_target_positive"
FEATURES = [c for c in df.columns if c not in OUTCOME_COLS + [ID_COL, TARGET]]

In [ ]:
print(f"shape: {df.shape}")
print(f"row count: {len(df):,}")
print(f"feature count: {len(FEATURES)} -> {FEATURES}")
print(f"unique content_id: {df[ID_COL].nunique():,} (one row = one content item: {df[ID_COL].is_unique})")
print("\nTarget distribution:")
print(df[TARGET].value_counts().rename({0: "0 (not positive)", 1: "1 (positive)"}))
print(f"\nPositive rate: {df[TARGET].mean():.1%}")

assert df[ID_COL].is_unique, "Each row must be a unique content item"
assert set(df[TARGET].unique()) <= {0, 1}, "Target must be binary"

In [ ]:
df.head(5)

In [ ]:
# Why accuracy is insufficient: a do-nothing baseline vs. a ranking metric
y = df[TARGET]
always_zero = np.zeros(len(y), dtype=int)
random_scores = rng.random(len(y))

print(f"Always-predict-0 accuracy : {accuracy_score(y, always_zero):.3f}  (looks good, catches 0 positives)")
print(f"Always-predict-0 ROC-AUC  : {roc_auc_score(y, always_zero):.3f}  (no better than chance)")
print(f"Random scores ROC-AUC     : {roc_auc_score(y, random_scores):.3f}")
print(f"Random scores PR-AUC      : {average_precision_score(y, random_scores):.3f}  (baseline = positive rate {y.mean():.3f})")

## 5. Why ML beats a fixed rule here

A fixed rule, such as `IF word_count > 1500 AND num_headings >= 5 THEN positive`, treats each feature as an independent on/off switch with a hand-picked cutoff. It cannot weigh features against each other (a shorter guide with strong backlinks may outperform a long page with none), it ignores interactions such as content type with length, and it gives only a hard yes/no with no sense of confidence. Rules also go stale: when search behavior or content norms shift, someone must notice and manually rewrite the thresholds.

An ML classifier learns these relationships from labelled history. It combines many features at once, captures non-linear effects and interactions, and outputs a **probability** that can be ranked and thresholded to match business capacity (for example, review the top 100 items this week). Its performance can also be measured objectively with ROC-AUC and PR-AUC on held-out data, so we can tell whether it is better than a rule rather than assuming so.

ML is not automatically better: with very little data, no signal, or a rule that is already well understood and stable, a simple rule is cheaper and easier to explain. Here, however, the outcome depends on many interacting signals and the data volume is sufficient, so a learned model is the stronger choice. A fixed rule remains useful as the **baseline** the model must beat.

## 6. Self-check

- [x] **Task type** stated clearly: supervised binary classification, with what is classified and why
- [x] **Target column** named (`is_target_positive`) with `1` vs `0` defined
- [x] **Proxy rationale** explained, including its limitations
- [x] **Success metrics** specified: ROC-AUC and PR-AUC, with a justification for not relying on accuracy
- [x] **Real dataframe** built with `pandas`; `shape`, row count, and feature count printed
- [x] **`.head(5)`** shown, demonstrating one row = one unique content item (`content_id` uniqueness asserted)
- [x] **Target created** in code (`is_target_positive`), with outcome columns excluded from features (no leakage)
- [x] **ML vs fixed rule** argument written (2-3 paragraphs)
- [x] Notebook runs top to bottom without errors